In [ ]:
%pip install -q -U "transformers>=4.46,<4.57" "peft>=0.13,<0.18" "accelerate>=1.0" "datasets>=2.20" "bitsandbytes>=0.43" pandas matplotlib

### Configuration
All hyper-parameters that must stay **identical across runs** live here. `MAX_TRAIN` can be
lowered (e.g. `os.environ["MAX_TRAIN"]="900"`) if you are short on Colab time — but then use the
same value for *every* run, and state it in the report.

In [ ]:
import os, gc, json, math, re, time, random, glob
import numpy as np
import pandas as pd
import torch

USE_DRIVE = True          
ROOT = os.getcwd()
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/AssignmentBeta_Q2"
    except Exception as e:  # fall back to local disk
        print("Drive not mounted, using local disk:", e)
os.makedirs(ROOT, exist_ok=True)
DATA_DIR    = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
ADAPTER_DIR = os.path.join(ROOT, "adapters")
for d in (DATA_DIR, RESULTS_DIR, ADAPTER_DIR):
    os.makedirs(d, exist_ok=True)
print("ROOT =", ROOT)

# ---------- experiment constants (identical for all runs) ----------
MODEL_ID      = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID    = "glaiveai/glaive-function-calling-v2"
SEED          = 42
N_TOTAL       = 2000
N_TEST        = 200
MAX_TRAIN     = int(os.environ.get("MAX_TRAIN", N_TOTAL - N_TEST))  # <= 1800
MAX_LEN       = 1024
EPOCHS        = 1
LR            = 2e-4
LR_SCHED      = "cosine"
WARMUP_RATIO  = 0.05
BATCH_SIZE    = 4
GRAD_ACCUM    = 4          # effective batch = 16
LORA_ALPHA_MULT = 2        # lora_alpha = 2 * r  (keeps alpha/r constant across the rank sweep)
LORA_DROPOUT  = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
MAX_NEW_TOKENS = 256
EVAL_BATCH    = 16

assert torch.cuda.is_available(), "Select a GPU runtime (Runtime -> Change runtime type -> T4 GPU)"
GPU_NAME = torch.cuda.get_device_name(0)
# T4 (sm_75) has no native bf16 -> use fp16; Ampere+ (sm_80+) -> bf16
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"GPU: {GPU_NAME} | compute dtype: {DTYPE} | MAX_TRAIN={MAX_TRAIN}")

def set_all_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    from transformers import set_seed
    set_seed(seed)
set_all_seeds()

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

##  LoRA arithmetic


In [ ]:
d, k, r = 4096, 4096, 16
params_B = d * r
params_A = r * k
lora_params = params_A + params_B
orig = d * k
print(f"A: {r} x {k} = {params_A:,}")
print(f"B: {d} x {r} = {params_B:,}")
print(f"(a) LoRA trainable params = r(d+k) = {r} x {d+k} = {lora_params:,}")
print(f"    original W0 params     = {d} x {k} = {orig:,}")
print(f"(b) percentage = {lora_params:,} / {orig:,} = {100*lora_params/orig:.5f} %")
print(f"    compression factor = {orig/lora_params:.0f}x fewer trainable parameters")

In [ ]:
torch.manual_seed(0)
d_, k_, r_ = 512, 512, 16
W0 = torch.randn(d_, k_) / math.sqrt(k_)
x = torch.randn(64, k_)
A_rand = torch.randn(r_, k_) / math.sqrt(k_)          # kaiming-like
B_zero = torch.zeros(d_, r_)
B_rand = torch.randn(d_, r_) / math.sqrt(r_)
scale = (2 * r_) / r_
base_out = x @ W0.T
out_lora = x @ (W0 + scale * B_zero @ A_rand).T
out_both = x @ (W0 + scale * B_rand @ A_rand).T
rel = lambda o: ((o - base_out).norm() / base_out.norm()).item()
print(f"relative output change at step 0, B=0  : {rel(out_lora):.4f}  (model == pretrained)")
print(f"relative output change at step 0, both random: {rel(out_both):.4f}  (pretrained function perturbed)")

In [ ]:
HID, LAYERS, KV = 1536, 28, 256
shapes = {"q_proj": (HID, HID), "k_proj": (HID, KV), "v_proj": (HID, KV), "o_proj": (HID, HID)}  # (in, out)
def lora_count(r, dora=False, adalora=False):
    per_layer = 0
    for (i, o) in shapes.values():
        per_layer += r * (i + o)
        if dora: per_layer += o         # magnitude vector
        if adalora: per_layer += r      # singular values Lambda
    return per_layer * LAYERS
for name, n in [("LoRA r=8", lora_count(8)), ("QLoRA r=8", lora_count(8)),
                ("DoRA r=8", lora_count(8, dora=True)), ("AdaLoRA init_r=12", lora_count(12, adalora=True)),
                ("LoRA r=4", lora_count(4)), ("LoRA r=16", lora_count(16)), ("LoRA r=32", lora_count(32))]:
    print(f"{name:20s} {n:>12,}")

##  Data preparation


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
IM_END = "<|im_end|>"
STOP_IDS = list({tokenizer.convert_tokens_to_ids(IM_END), tokenizer.eos_token_id, tokenizer.pad_token_id})

SYSTEM_TEMPLATE = (
    "You are a customer-support function-calling assistant. You have access to these functions:\n"
    "{schemas}\n\n"
    "Reply with ONLY one JSON object of the form "
    '{{"name": "<function name>", "arguments": {{...}}}} and nothing else.'
)

_dec = json.JSONDecoder()

def parse_schemas(system_text):
    # Extract every top-level JSON object from the glaive system prompt.
    out, i = [], system_text.find("{")
    while 0 <= i < len(system_text):
        try:
            obj, end = _dec.raw_decode(system_text, i)
            if isinstance(obj, dict) and "name" in obj:
                out.append(obj)
            i = system_text.find("{", end)
        except json.JSONDecodeError:
            i = system_text.find("{", i + 1)
    return out

def parse_functioncall(fc):
    # Parse glaive '<functioncall> {"name": "x", "arguments": '{...}'}' -> (name, args dict) or None.
    fc = fc.strip()
    m = re.search(r'"name"\s*:\s*"([^"]+)"', fc)
    if not m:
        return None
    name = m.group(1)
    am = re.search(r'"arguments"\s*:\s*\'(.*)\'\s*\}\s*$', fc, re.S)
    args = None
    if am:
        try:
            args = json.loads(am.group(1))
        except json.JSONDecodeError:
            return None
    else:  # sometimes arguments is a plain JSON object / string
        try:
            obj = json.loads(fc)
            args = obj.get("arguments")
            if isinstance(args, str):
                args = json.loads(args)
        except Exception:
            return None
    if not isinstance(args, dict):
        return None
    return name, args

CHAT_RE = re.compile(r"^\s*USER:\s*(.*?)\s*ASSISTANT:\s*(.*?)(?:<\|endoftext\|>|\n\n\nUSER:|\n\n\nFUNCTION RESPONSE:|$)", re.S)

def build_prompt_messages(schemas, user_msg):
    sys_txt = SYSTEM_TEMPLATE.format(schemas="\n".join(json.dumps(s) for s in schemas))
    return [{"role": "system", "content": sys_txt}, {"role": "user", "content": user_msg}]

def convert_row(row):
    schemas = parse_schemas(row["system"])
    if not schemas:
        return None
    m = CHAT_RE.match(row["chat"])
    if not m:
        return None
    user_msg, reply = m.group(1).strip(), m.group(2).strip()
    if not reply.startswith("<functioncall>") or not user_msg:
        return None
    parsed = parse_functioncall(reply[len("<functioncall>"):])
    if parsed is None:
        return None
    name, args = parsed
    names = [s["name"] for s in schemas]
    if name not in names:
        return None
    target = json.dumps({"name": name, "arguments": args}, ensure_ascii=False)
    return {"user": user_msg, "schemas": schemas, "function_names": names,
            "target": target, "gold_name": name, "gold_args": args}

def n_tokens(ex):
    p = tokenizer.apply_chat_template(build_prompt_messages(ex["schemas"], ex["user"]), tokenize=True, add_generation_prompt=True)
    t = tokenizer(ex["target"] + IM_END, add_special_tokens=False)["input_ids"]
    return len(p) + len(t)

def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r_ in rows:
            f.write(json.dumps(r_, ensure_ascii=False) + "\n")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

train_path, test_path = os.path.join(DATA_DIR, "train.jsonl"), os.path.join(DATA_DIR, "test.jsonl")
if os.path.exists(train_path) and os.path.exists(test_path):
    train_rows, test_rows = load_jsonl(train_path), load_jsonl(test_path)
    print("Loaded cached splits")
else:
    from datasets import load_dataset
    raw = load_dataset(DATASET_ID, split="train").shuffle(seed=SEED)
    kept, scanned = [], 0
    for row in raw:
        scanned += 1
        ex = convert_row(row)
        if ex is not None and n_tokens(ex) <= MAX_LEN:
            kept.append(ex)
            if len(kept) == N_TOTAL:
                break
    print(f"scanned {scanned:,} rows -> kept {len(kept):,} clean function-call examples")
    rng = random.Random(SEED)
    rng.shuffle(kept)
    test_rows, train_rows = kept[:N_TEST], kept[N_TEST:]
    save_jsonl(train_rows, train_path); save_jsonl(test_rows, test_path)
train_rows = train_rows[:MAX_TRAIN]
print(f"train={len(train_rows)}  test={len(test_rows)}")
print("\nExample prompt user msg:", test_rows[0]["user"][:200])
print("Example target:", test_rows[0]["target"])

### Tokenisation with loss masking
Each example is rendered with the Qwen2.5 chat template: `system + user + assistant-generation-prompt`
→ **prompt tokens (labels = -100, no loss)**, then `target JSON + <|im_end|>` → **target tokens
(loss computed)**. Thus the model is trained only to produce the JSON call and the stop token
(which teaches it to stop instead of rambling).

In [ ]:
from datasets import Dataset

def tokenize_example(ex):
    p_ids = tokenizer.apply_chat_template(build_prompt_messages(ex["schemas"], ex["user"]), tokenize=True, add_generation_prompt=True)
    t_ids = tokenizer(ex["target"] + IM_END, add_special_tokens=False)["input_ids"]
    if len(p_ids) + len(t_ids) > MAX_LEN:      # safety: trim prompt from the left (never the target)
        p_ids = p_ids[-(MAX_LEN - len(t_ids)):]
    ids = p_ids + t_ids
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": [-100] * len(p_ids) + t_ids}

train_ds = Dataset.from_list([tokenize_example(e) for e in train_rows])
lens = [len(x) for x in train_ds["input_ids"]]
print(f"train tokens/example: mean={np.mean(lens):.0f}, max={max(lens)}")

class PadCollator:
    def __init__(self, pad_id): self.pad_id = pad_id
    def __call__(self, feats):
        L = max(len(f["input_ids"]) for f in feats)
        ids  = [f["input_ids"] + [self.pad_id] * (L - len(f["input_ids"])) for f in feats]
        att  = [f["attention_mask"] + [0] * (L - len(f["attention_mask"])) for f in feats]
        labs = [f["labels"] + [-100] * (L - len(f["labels"])) for f in feats]
        return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(att), "labels": torch.tensor(labs)}
collator = PadCollator(tokenizer.pad_token_id)

# steps per run (needed by AdaLoRA's budget scheduler)
TOTAL_STEPS = math.ceil(math.ceil(len(train_ds) / BATCH_SIZE) / GRAD_ACCUM) * EPOCHS
print("optimizer steps per run:", TOTAL_STEPS)

### Evaluation
Per-example predictions are saved to `results/preds_<run>.csv` for error analysis.

In [ ]:
def extract_call(text):
    # Lenient extraction of the first JSON object in the model output.
    t = text.strip()
    t = re.sub(r"^```(?:json)?", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    i = t.find("{")
    while i != -1:
        try:
            obj, _ = _dec.raw_decode(t, i)
            if isinstance(obj, dict):
                if "function" in obj and "name" not in obj and isinstance(obj["function"], dict):
                    obj = obj["function"]
                args = obj.get("arguments", obj.get("parameters"))
                if isinstance(args, str):
                    try: args = json.loads(args)
                    except Exception: pass
                return obj.get("name"), args
        except json.JSONDecodeError:
            pass
        i = t.find("{", i + 1)
    return None, None

def strict_valid(text):
    try:
        return isinstance(json.loads(text.strip()), dict)
    except Exception:
        return False

@torch.no_grad()
def generate_outputs(model, rows, batch_size=EVAL_BATCH, max_new_tokens=MAX_NEW_TOKENS):
    tokenizer.padding_side = "left"
    model.eval()
    prompts = [tokenizer.apply_chat_template(build_prompt_messages(e["schemas"], e["user"]), tokenize=False, add_generation_prompt=True) for e in rows]
    outs = []
    torch.cuda.synchronize(); t0 = time.time()
    for i in range(0, len(prompts), batch_size):
        enc = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None,
                             eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id, use_cache=True)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    torch.cuda.synchronize(); elapsed = time.time() - t0
    tokenizer.padding_side = "right"
    return outs, elapsed

def score(rows, outs):
    recs = []
    for e, o in zip(rows, outs):
        name, args = extract_call(o)
        recs.append({"user": e["user"], "function_names": json.dumps(e["function_names"]),
                     "gold": e["target"], "output": o,
                     "json_valid": strict_valid(o),
                     "tool_correct": name == e["gold_name"],
                     "full_match": (name == e["gold_name"]) and (args == e["gold_args"]),
                     "pred_name": name, "pred_args": json.dumps(args, ensure_ascii=False) if args is not None else None})
    df = pd.DataFrame(recs)
    return df, {"json_valid": int(df.json_valid.sum()), "tool_acc": int(df.tool_correct.sum()), "full_match": int(df.full_match.sum())}

def evaluate(model, rows, tag):
    outs, elapsed = generate_outputs(model, rows)
    df, m = score(rows, outs)
    df.to_csv(os.path.join(RESULTS_DIR, f"preds_{tag}.csv"), index=False)
    m["avg_latency_s"] = round(elapsed / len(rows), 4)
    print(f"[{tag}] JSON valid {m['json_valid']}/{len(rows)} | tool acc {m['tool_acc']}/{len(rows)} | full match {m['full_match']}/{len(rows)} | {m['avg_latency_s']}s/ex")
    return m

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback
from peft import LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()
    try: torch.cuda.ipc_collect()
    except Exception: pass

def load_base(quant4=False):
    kw = dict(torch_dtype=DTYPE, device_map={"": 0}, attn_implementation="sdpa")
    if quant4:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)

class AdaLoraCallback(TrainerCallback):
    # AdaLoRA must re-estimate importance & re-allocate rank after every optimizer step.
    def __init__(self, peft_model): self.peft_model = peft_model
    def on_optimizer_step(self, args, state, control, **kwargs):
        self.peft_model.base_model.update_and_allocate(state.global_step)

def adalora_effective_params(model):
    # Params actually in use after AdaLoRA pruning (non-zero singular values).
    total, ranks = 0, []
    for _, m in model.named_modules():
        if hasattr(m, "lora_E") and "default" in getattr(m, "lora_E", {}):
            E = m.lora_E["default"]
            r_eff = int((E.detach() != 0).sum().item())
            i_ = m.lora_A["default"].shape[1]; o_ = m.lora_B["default"].shape[0]
            total += r_eff * (i_ + o_) + r_eff; ranks.append(r_eff)
    return total, ranks

def run_name(method, r): return f"{method}_r{r}"

def run_experiment(method, r=8, force=False):
    name = run_name(method, r)
    res_path = os.path.join(RESULTS_DIR, f"{name}.json")
    if os.path.exists(res_path) and not force:
        print(f"skip {name} (results exist)")
        return json.load(open(res_path))
    free_gpu(); set_all_seeds()
    torch.cuda.reset_peak_memory_stats()

    model = load_base(quant4=(method == "qlora"))
    model.config.use_cache = False
    if method == "qlora":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.enable_input_require_grads()

    common = dict(lora_alpha=LORA_ALPHA_MULT * r, lora_dropout=LORA_DROPOUT, target_modules=TARGET_MODULES,
                  bias="none", task_type="CAUSAL_LM")
    if method in ("lora", "qlora"):
        cfg = LoraConfig(r=r, **common)
    elif method == "dora":
        cfg = LoraConfig(r=r, use_dora=True, **common)
    elif method == "adalora":
        init_r = int(round(1.5 * r))                      # r=8 -> init 12, target 8 (assignment spec)
        tinit, tfinal = max(1, int(0.10 * TOTAL_STEPS)), max(1, int(0.20 * TOTAL_STEPS))
        cfg = AdaLoraConfig(init_r=init_r, target_r=r, tinit=tinit, tfinal=tfinal, deltaT=max(1, TOTAL_STEPS // 50),
                            beta1=0.85, beta2=0.85, orth_reg_weight=0.5, total_step=TOTAL_STEPS, **common)
    else:
        raise ValueError(method)
    set_all_seeds()                                       # identical adapter initialisation RNG
    model = get_peft_model(model, cfg)
    for p in model.parameters():                          # keep trainable adapter weights in fp32 (AMP-safe)
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    trainable, total = model.get_nb_trainable_parameters()
    print(f"{name}: trainable {trainable:,} / {total:,} = {100*trainable/total:.4f}%")

    args = TrainingArguments(
        output_dir=os.path.join(ROOT, "tmp_trainer", name), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, lr_scheduler_type=LR_SCHED, warmup_ratio=WARMUP_RATIO, weight_decay=0.0,
        max_grad_norm=1.0, fp16=not USE_BF16, bf16=USE_BF16, logging_steps=10, save_strategy="no",
        report_to="none", seed=SEED, data_seed=SEED, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}, remove_unused_columns=False,
        label_names=["labels"], dataloader_num_workers=0,
        optim="paged_adamw_32bit" if method == "qlora" else "adamw_torch",
    )
    callbacks = [AdaLoraCallback(model)] if method == "adalora" else []
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator, callbacks=callbacks)

    torch.cuda.synchronize(); t0 = time.time()
    out = trainer.train()
    torch.cuda.synchronize(); train_min = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{name}: train {train_min:.1f} min | peak mem {peak_gb:.2f} GB | loss {out.training_loss:.4f}")

    extra = {}
    if method == "adalora":
        eff, ranks = adalora_effective_params(model)
        extra = {"adalora_effective_params": eff, "adalora_rank_hist": {int(k_): ranks.count(k_) for k_ in sorted(set(ranks))}}
        print("AdaLoRA effective params after pruning:", f"{eff:,}", "| rank histogram:", extra["adalora_rank_hist"])

    model.save_pretrained(os.path.join(ADAPTER_DIR, name))
    log_hist = [h for h in trainer.state.log_history if "loss" in h]
    # switch to inference mode
    model.config.use_cache = True
    try: model.base_model.model.gradient_checkpointing_disable()
    except Exception: pass
    metrics = evaluate(model, test_rows, name)

    res = {"run": name, "method": method, "r": r, "trainable_params": int(trainable), "total_params": int(total),
           "trainable_pct": round(100 * trainable / total, 4), "peak_mem_gb": round(peak_gb, 3),
           "train_time_min": round(train_min, 2), "final_train_loss": round(out.training_loss, 4),
           **metrics, "n_test": len(test_rows), "n_train": len(train_ds), "gpu": GPU_NAME, "dtype": str(DTYPE),
           "loss_curve": [(h["step"], h["loss"]) for h in log_hist], **extra}
    json.dump(res, open(res_path, "w"), indent=2)
    del trainer, model; free_gpu()
    return res

### Row 0 — base model, no fine-tuning

In [ ]:
def run_base(force=False):
    res_path = os.path.join(RESULTS_DIR, "base_r0.json")
    if os.path.exists(res_path) and not force:
        print("skip base (results exist)"); return json.load(open(res_path))
    free_gpu(); torch.cuda.reset_peak_memory_stats()
    model = load_base()
    total = sum(p.numel() for p in model.parameters())
    m = evaluate(model, test_rows, "base_r0")
    res = {"run": "base_r0", "method": "base", "r": 0, "trainable_params": 0, "total_params": int(total),
           "trainable_pct": 0.0, "peak_mem_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
           "train_time_min": 0.0, **m, "n_test": len(test_rows), "gpu": GPU_NAME, "dtype": str(DTYPE)}
    json.dump(res, open(res_path, "w"), indent=2)
    del model; free_gpu()
    return res

base_res = run_base()

In [ ]:
MAIN_RUNS = [("lora", 8), ("qlora", 8), ("dora", 8), ("adalora", 8)]
for method, r in MAIN_RUNS:
    print("=" * 80); print("RUN", method, r)
    run_experiment(method, r)

In [ ]:
import matplotlib.pyplot as plt

COLS = ["run", "method", "r", "trainable_params", "trainable_pct", "peak_mem_gb", "train_time_min",
        "json_valid", "tool_acc", "full_match", "avg_latency_s", "final_train_loss"]

def load_results():
    rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(RESULTS_DIR, "*_r*.json")))]
    return pd.DataFrame(rows)

all_res = load_results()
order = ["base_r0", "lora_r8", "qlora_r8", "dora_r8", "adalora_r8"]
main = all_res.set_index("run").reindex([o for o in order if o in set(all_res.run)]).reset_index()
main_tbl = main[[c for c in COLS if c in main.columns]].copy()
for c in ["json_valid", "tool_acc", "full_match"]:
    main_tbl[c + "_%"] = (100 * main_tbl[c] / N_TEST).round(1)
main_tbl.to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
all_res[[c for c in COLS if c in all_res.columns]].to_csv(os.path.join(RESULTS_DIR, "all_runs.csv"), index=False)
display(main_tbl)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
labels = main_tbl.run.tolist()
for ax, (col, title) in zip(axes.flat, [("json_valid", "Strict JSON validity (/200)"), ("tool_acc", "Tool accuracy (/200)"),
                                         ("full_match", "Full match (/200)"), ("peak_mem_gb", "Peak GPU memory (GB)"),
                                         ("train_time_min", "Training time (min)"), ("avg_latency_s", "Inference latency (s / example)")]):
    vals = main_tbl[col].values
    bars = ax.bar(labels, vals, color=["grey", "tab:blue", "tab:green", "tab:orange", "tab:purple"][:len(labels)])
    ax.bar_label(bars, fmt="%.2f" if main_tbl[col].dtype.kind == "f" else "%d", fontsize=8)
    ax.set_title(title); ax.tick_params(axis="x", rotation=30)
plt.suptitle(f"Qwen2.5-1.5B-Instruct — PEFT comparison on glaive-function-calling-v2 ({GPU_NAME})")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "comparison.png"), dpi=150); plt.show()

# training loss curves
plt.figure(figsize=(7, 4))
for _, row in main.iterrows():
    if isinstance(row.get("loss_curve"), list) and row["loss_curve"]:
        s, l = zip(*row["loss_curve"]); plt.plot(s, l, label=row["run"])
plt.xlabel("optimizer step"); plt.ylabel("train loss"); plt.legend(); plt.title("Training loss")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "loss_curves.png"), dpi=150); plt.show()

##  Rank sweep for the best method


In [ ]:
ft = main[main.method != "base"].copy()
ft = ft.sort_values(["full_match", "tool_acc", "json_valid"], ascending=False)
BEST_METHOD = ft.iloc[0]["method"]
print("Best method from Section 2:", BEST_METHOD)
# BEST_METHOD = "lora"   # <- uncomment to override manually

for r in [4, 16, 32]:
    print("=" * 80); print("RANK SWEEP", BEST_METHOD, r)
    run_experiment(BEST_METHOD, r)

In [ ]:
all_res = load_results()
sweep = all_res[all_res.method == BEST_METHOD].sort_values("r")
sweep_tbl = sweep[[c for c in COLS if c in sweep.columns]]
sweep_tbl.to_csv(os.path.join(RESULTS_DIR, "rank_sweep.csv"), index=False)
display(sweep_tbl)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(sweep.r, 100 * sweep.tool_acc / N_TEST, "o-", label="tool accuracy")
ax[0].plot(sweep.r, 100 * sweep.full_match / N_TEST, "s--", label="full match")
ax[0].plot(sweep.r, 100 * sweep.json_valid / N_TEST, "^:", label="strict JSON valid")
base_tool = 100 * main.loc[main.method == "base", "tool_acc"].values[0] / N_TEST if (main.method == "base").any() else None
if base_tool is not None:
    ax[0].axhline(base_tool, color="grey", ls="--", lw=1, label="base model tool acc")
ax[0].set_xscale("log", base=2); ax[0].set_xticks(sweep.r); ax[0].set_xticklabels(sweep.r)
ax[0].set_xlabel("LoRA rank r (log2 scale)"); ax[0].set_ylabel("% of 200 test examples")
ax[0].set_title(f"{BEST_METHOD}: accuracy vs rank"); ax[0].legend(); ax[0].grid(alpha=.3)
for x, y in zip(sweep.r, 100 * sweep.tool_acc / N_TEST): ax[0].annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center")
ax[1].plot(sweep.r, sweep.peak_mem_gb, "o-", c="tab:red", label="peak memory (GB)")
ax2 = ax[1].twinx(); ax2.plot(sweep.r, sweep.trainable_params / 1e6, "s--", c="tab:blue", label="trainable params (M)")
ax[1].set_xscale("log", base=2); ax[1].set_xticks(sweep.r); ax[1].set_xticklabels(sweep.r)
ax[1].set_xlabel("rank r"); ax[1].set_ylabel("peak GPU memory (GB)"); ax2.set_ylabel("trainable params (M)")
ax[1].set_title("Cost vs rank"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "rank_sweep.png"), dpi=150); plt.show()

ta = (100 * sweep.tool_acc / N_TEST).round(1).tolist()
print("tool accuracy by rank:", dict(zip(sweep.r.tolist(), ta)))
print("change per doubling:", [round(b - a, 1) for a, b in zip(ta, ta[1:])], "percentage points")
print("(With n=200, one example = 0.5 pp; differences of ~1-2 pp are within noise: the 95% CI half-width at 90% acc is ~4 pp.)")

In [ ]:
best_run = ft.iloc[0]["run"]
sw = load_results(); cand = sw[sw.method != "base"].sort_values(["full_match", "tool_acc"], ascending=False)
best_run = cand.iloc[0]["run"]          # best over all runs incl. rank sweep
print("Error analysis on", best_run)
preds = pd.read_csv(os.path.join(RESULTS_DIR, f"preds_{best_run}.csv"))
test_by_user = {e["user"]: e for e in test_rows}

def label_error(row):
    e = test_by_user.get(row.user)
    name, args = extract_call(str(row.output))
    if name is None:
        return "refused to call tool" if "{" not in str(row.output) else "broken JSON"
    if name != e["gold_name"]:
        return "wrong tool (invented name)" if name not in e["function_names"] else "wrong tool"
    if not isinstance(args, dict):
        return "broken JSON"
    schema = next((s for s in e["schemas"] if s["name"] == name), {})
    allowed = set(schema.get("parameters", {}).get("properties", {}).keys()) | set(e["gold_args"].keys())
    if set(args) - allowed:
        return "made-up argument"
    msg = e["user"].lower()
    for k_, v in args.items():
        if k_ not in e["gold_args"] and isinstance(v, (str, int, float)) and str(v).lower() not in msg:
            return "made-up argument"
    return "other (wrong/missing argument value)"

wrong = preds[~preds.full_match].copy()
wrong["label"] = wrong.apply(label_error, axis=1)
print(f"{len(wrong)} wrong examples; label distribution:")
print(wrong.label.value_counts())
wrong.to_csv(os.path.join(RESULTS_DIR, f"errors_{best_run}.csv"), index=False)

# pick 5 examples, trying to cover different labels
picked = pd.concat([g.head(1) for _, g in wrong.groupby("label")])
picked = pd.concat([picked, wrong.drop(picked.index)]).head(5)
for i, row in enumerate(picked.itertuples(), 1):
    print(f"\n--- Example {i}: {row.label} ---")
    print("Customer message:", row.user[:400])
    print("Correct call    :", row.gold)
    print("Model output    :", str(row.output)[:400])
picked[["user", "gold", "output", "label"]].to_csv(os.path.join(RESULTS_DIR, "five_errors.csv"), index=False)

In [ ]:
N_MERGE = 50
merge_rows = test_rows[:N_MERGE]
merge_results = []

def timed_eval(model, tag):
    outs, el = generate_outputs(model, merge_rows)
    _, m = score(merge_rows, outs)
    rec = {"setting": tag, "latency_s_per_ex": round(el / N_MERGE, 4), "full_match": m["full_match"], "tool_acc": m["tool_acc"],
           "json_valid": m["json_valid"], "mem_gb": round(torch.cuda.memory_allocated() / 1024**3, 2)}
    print(rec); merge_results.append(rec); return outs

for method in ["lora", "dora"]:
    adir = os.path.join(ADAPTER_DIR, run_name(method, 8))
    if not os.path.exists(adir): print("missing", adir); continue
    free_gpu()
    m_ = PeftModel.from_pretrained(load_base(), adir)
    o_unmerged = timed_eval(m_, f"{method} unmerged (fp16 base + adapter)")
    m_ = m_.merge_and_unload()
    o_merged = timed_eval(m_, f"{method} merged into fp16 weights")
    print(f"  identical outputs merged vs unmerged: {sum(a == b for a, b in zip(o_unmerged, o_merged))}/{N_MERGE}")
    del m_; free_gpu()

adir = os.path.join(ADAPTER_DIR, run_name("qlora", 8))
if os.path.exists(adir):
    free_gpu()
    m_ = PeftModel.from_pretrained(load_base(quant4=True), adir)
    timed_eval(m_, "qlora unmerged (NF4 base + adapter)")
    del m_; free_gpu()
    m_ = PeftModel.from_pretrained(load_base(), adir).merge_and_unload()   # adapter merged into ORIGINAL fp16 weights
    timed_eval(m_, "qlora adapter merged into fp16 (de-quantised) base")
    del m_; free_gpu()

merge_df = pd.DataFrame(merge_results)
merge_df.to_csv(os.path.join(RESULTS_DIR, "merge_inference.csv"), index=False)
display(merge_df)

## Final summary 

In [ ]:
print("== Section 2 table (results/summary.csv) ==")
print(main_tbl.to_string(index=False))
print("\n== Section 3 rank sweep (results/rank_sweep.csv) ==")
print(sweep_tbl.to_string(index=False))
print("\n== Merge / inference (results/merge_inference.csv) ==")
print(merge_df.to_string(index=False))
print("\nFiles in", RESULTS_DIR); print(sorted(os.listdir(RESULTS_DIR)))